# Gemini LLM API

Python 코드에서 LLM에게 요청을 보내고 응답을 사용하는 기본 흐름을 익힙니다.

공식 문서: https://ai.google.dev/gemini-api/docs/get-started

## 실습 환경 준비

패키지를 설치합니다. 설치 후 import 오류가 계속되면 커널을 한 번 재시작하세요.

`pip install google-genai`

### API 키 준비

프로젝트 폴더의 `.env` 파일에 다음과 같이 적습니다.

```text
GEMINI_API_KEY=발급받은_API_키
```

API 키는 비밀번호와 같습니다. 코드에 직접 적거나 Git에 올리지 마세요. `.gitignore`에 `.env`가 포함되어 있는지도 확인합니다.

In [2]:
import os
from pprint import pprint

import requests
from dotenv import load_dotenv
from google import genai
from google.genai import errors

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError(".env 파일에 GEMINI_API_KEY를 설정해 주세요.")

# 모델 이름은 바뀔 수 있으므로 환경 변수로 교체할 수 있게 둡니다.
model = os.getenv("GEMINI_MODEL", "gemini-3.6-flash")
print("준비 완료 / 사용 모델:", model)


준비 완료 / 사용 모델: gemini-3.6-flash


## REST API로 첫 요청 보내기

SDK를 사용하기 전에 HTTP 요청의 구성 요소를 직접 확인합니다. `requests.post()`가 URL로 요청을 보내고, headers에는 인증 정보, body에는 모델이 처리할 데이터를 담습니다.

- URL: 요청을 받을 API 주소
- headers: 데이터 형식과 API 키
- body: 모델, 입력, 요청 설정
- response: 상태 코드와 응답 body

In [3]:
url = "https://generativelanguage.googleapis.com/v1beta/interactions"
headers = {
    "Content-Type": "application/json",
    "x-goog-api-key": api_key,
}
body = {
    "model": model,
    "input": "LLM API를 30자 이내의 한 문장으로 설명해줘.",
    "store": False,
}

response = requests.post(url, headers=headers, json=body, timeout=30)
print("HTTP 상태 코드:", response.status_code)
response.raise_for_status()

response_data = response.json()
pprint(response_data)

# 기본 텍스트 요청에서는 마지막 step에 모델의 최종 답변이 들어 있습니다.
answer = response_data["steps"][-1]["content"][0]["text"]
print("\n최종 답변:")
print(answer)

HTTP 상태 코드: 200
{'created': '2026-08-25T06:11:05Z',
 'model': 'gemini-3.6-flash',
 'object': 'interaction',
 'service_tier': 'standard',
 'status': 'completed',
 'steps': [{'signature': 'Eu0sCuosARFNMg+XsPjYJvIb66gKtXoO6g7u5K62EcTQr7cLfQllH9ygiKXgYZt+iQkgo6pc3DGrt03MTblFin+AN838KOQZCa/jo9JVMn1sayvVSOOTR4ZyIn8RKAolhwdAjQ8VkaJiTjwH3M5R5WfQHg1eC4VgOyfCKrc8j3dyVP8lz06Xw2VxHjEEJ/AwWrNAl6EJhC6B/KvLZGEYUpyVYJxO4CQdY7Jg7IMn+EO2WZm/98x4YyZFFCEZOvVrrvBS1+yUABrMQbdap9RTyk3Lo0WH3UdWaHYRfaUwVTQLbvkhmxjt1oQAhzk4nMZK3QjT1HoEopBL2Bpsuuv4JwK+quFxC5d+PhHb3Jp+dtikWhNxq7TaqpXrehBuKp4g76DqhHy3rkGvqpjT01/P6N0tdR4sSsY240QpbNZ5bSetkbjhBsRnusZDFGgOh/7d0MUYhMfYZdI/eJ+4GUxxAbm/3bhPF48uYCThygKf/4CgKJbeN3srQaW6Krb+OoTJVG9YHKdae/hMrzQjRWIpD/hyp+bib6znCsqdpywxmwdMpv32XW8t+ZSR1XK9Y7U/sIa32u/VGb4hx8yWPuoW61/yJVFSarX+daQYognxvbzlfcsjXCmc/VSx7grFiUXIF3TK0rFkUzQPSljjIOB/X6HYWD3UX5xHrOQDRcpMAVLY9dtQ5GJR0gRtIq2N8qyCzyGVQPUYZicP+LpvplWi6+kmjaT6VI9kjrAl0Kvc/yoRvw8j8fIIf6Jdt3azao+UH3SNn3RQw3mG4qKR17F+iLKuQbI4nNDLkGFr5XGGcENg7

REST 응답은 대략 다음과 같은 구조입니다. ID와 답변 내용은 실행할 때마다 달라질 수 있습니다.

```python
{
    "id": "interaction의 고유 ID",
    "status": "completed",
    "steps": [
        {"type": "user_input", "content": [...]},
        {
            "type": "model_output",
            "content": [
                {"type": "text", "text": "LLM API는 ..."}
            ],
        },
    ],
}
```


## SDK로 같은 요청 보내기

SDK는 앞에서 직접 작성한 URL, headers, JSON 변환과 응답 구조 처리를 편리하게 감싸 줍니다. HTTP 요청이 사라진 것이 아니라 라이브러리 안에서 대신 처리되는 것입니다.

- `model`: 사용할 모델
- `input`: 모델이 처리할 질문이나 데이터
- `output_text`: SDK가 최종 텍스트를 꺼내 제공하는 편의 속성

In [4]:
client = genai.Client(api_key=api_key)

interaction = client.interactions.create(
    model=model,
    input="LLM API를 30자 이내의 한 문장으로 설명해줘.",
    store=False,
)

print(interaction.output_text)


내 서비스에 AI 언어모델을 연결하는 통로입니다.


### SDK 코드와 응답 읽기

- `genai.Client(...)`: API 키를 가진 Gemini 클라이언트 객체를 만듭니다. 이후 요청은 이 객체를 통해 보냅니다.
- `client.interactions.create(...)`: 새 interaction 요청을 만들어 Gemini API로 전송합니다. 이름은 `create`이지만 로컬 객체만 만드는 것이 아니라 실제 네트워크 요청이 일어납니다.
- `interaction.output_text`: 여러 step으로 구성된 응답에서 최종 텍스트를 SDK가 꺼내 제공하는 편의 속성입니다.

`create()`가 반환하는 `Interaction` 객체는 개념적으로 다음 정보를 가집니다. 실제 객체를 dictionary처럼 대괄호로 읽는 것은 아니고 점(`.`)으로 속성에 접근합니다.

```python
interaction.id           # 요청의 고유 ID
interaction.status       # completed, failed 등의 처리 상태
interaction.steps        # 사용자 입력과 모델 출력 과정
interaction.output_text  # 마지막 모델 출력의 최종 텍스트
```

REST의 JSON 응답과 SDK의 `Interaction` 객체는 서로 다른 결과가 아니라, 같은 API 응답을 서로 다른 방식으로 읽는 것입니다.

### `input`에 넣을 수 있는 두 가지 형태

간단한 텍스트 요청에서는 `input`에 문자열 하나를 바로 넣을 수 있습니다.

```python
input="안녕하세요"
```

대화 기록처럼 입력의 역할과 종류를 구분해야 할 때는 dictionary로 만든 step의 list를 넣습니다.

```python
input=[
    {
        "type": "user_input",
        "content": [
            {"type": "text", "text": "안녕하세요"}
        ],
    }
]
```

여기서 `input` 자체가 `type`으로 바뀌는 것은 아닙니다. `input`에 들어가는 각 항목이 `type` 필드로 자신의 역할을 표시합니다. `user_input`은 사용자 입력 step, 그 안의 `text`는 텍스트 콘텐츠라는 뜻입니다.

| 상황 | `input` 형태 |
|---|---|
| 한 번의 간단한 질문 | 문자열 `str` |
| 텍스트 외 다른 종류를 구분 | `type`이 있는 dictionary의 `list` |

### 직접 바꿔 보기

질문의 주제, 답변 문장 수, 설명을 듣는 대상을 바꿔 다시 실행해 보세요.

**생각해 볼 질문:** 같은 질문을 다시 보내면 답변이 완전히 같나요? 다른 API들과는 무엇이 다른가요?

## System instruction과 사용자 입력

`system_instruction`은 답변 전반에 적용할 역할과 규칙이고, `input`은 이번에 처리할 실제 질문입니다. 역할, 작업, 제약, 출력 형식을 구체적으로 적으면 의도를 전달하기 쉽습니다.

> 프롬프트는 답변 품질을 유도하지만 사실성을 보장하지는 않습니다. 중요한 결과는 별도로 검증합니다.

In [5]:
question = "API 키를 코드에 직접 적으면 왜 위험해? 한 문장으로 답해줘."
instructions = {
    "친절한 강사": "Python 입문 강사입니다. 쉬운 말로 40자 이내에서 답하세요.",
    "보안 담당자": "보안 담당자입니다. 단호한 말투로 40자 이내에서 답하세요.",
}

for name, instruction in instructions.items():
    result = client.interactions.create(
        model=model, input=question,
        system_instruction=instruction, store=False,
    )
    print(f"--- {name} ---")
    print(result.output_text, "\n")

--- 친절한 강사 ---
코드가 노출되면 남이 내 키를 써서 요금 폭탄을 맞을 수 있어요. 

--- 보안 담당자 ---
코드 유출 즉시 해킹 및 데이터 탈취 등 대형 보안 사고로 이어집니다. 



## 대화 맥락 이어가기



첫 요청에서 이름을 알려줘도, 두 번째 요청에 이전 대화를 넣지 않으면 모델은 이름을 알 수 없습니다.

In [6]:
# 첫 번째 요청: 이름 알려주기
tell_name = client.interactions.create(
    model=model,
    input="내 이름은 민수야. '확인'이라고만 답해.",
    store=False,
)
print("첫 번째 답변:", tell_name.output_text)

# 두 번째 요청: 이전 대화를 전달하지 않고 이름 물어보기
ask_without_history = client.interactions.create(
    model=model,
    input="내 이름이 뭐야? 정보가 없으면 '모름'이라고만 답해.",
    store=False,
)
print("두 번째 답변:", ask_without_history.output_text)

첫 번째 답변: 확인
두 번째 답변: 모름


### history를 넣어 두 번 요청하기

이번에는 첫 요청의 입력과 응답을 history에 보존한 뒤 두 번째 질문과 함께 전달합니다.

In [7]:
# 첫 번째 요청: 이름 알려주기
history = [
    {
        "type": "user_input",
        "content": [
            {"type": "text", "text": "내 이름은 민수야. '확인'이라고만 답해."}
        ],
    }
]

turn1 = client.interactions.create(
    model=model, input=history, store=False,
)
print("첫 번째 답변:", turn1.output_text)

# 기존 user_input 뒤에 thought와 model_output을 추가합니다.
history.extend(
    step.model_dump(exclude_none=True) for step in turn1.steps
)

# 두 번째 요청: 이름 물어보기
history.append(
    {
        "type": "user_input",
        "content": [
            {"type": "text", "text": "내 이름이 뭐야? 이름만 답해."}
        ],
    }
)

print()
print("두 번째 요청에 보내는 대화 기록:")
pprint(history)

turn2 = client.interactions.create(
    model=model, input=history, store=False,
)
print("두 번째 답변:", turn2.output_text)

첫 번째 답변: 확인

두 번째 요청에 보내는 대화 기록:
[{'content': [{'text': "내 이름은 민수야. '확인'이라고만 답해.", 'type': 'text'}],
  'type': 'user_input'},
 {'signature': 'EoAFCv0EARFNMg+aR4giFBQtJlAUxuMq0uX9xw6KvNJFlryiVlED7hJT7NtY6oP2qOVCFsv3YlGpoEr3YNAXDBXe9EajkIYaXigBrjoLaqziyxj4YOoExXL8Zg4GiGWL+gGuFGRpyHHpEs48YBFJg6DH9XxBy4yT+2HEdHRmUXrU7ZJ4jkLm2a1CxwpQH2RfM4cck4Xvk9xCtMHbIxYgZAOXUpHKLX4doSjo25+SQM6ZJoGexQU5wEOe03np1X/wCQTEi4b4pZTjIJBYr6Yu4lmys80RoCifUmu8sZ7/A2+EmVClMVr6yz20fJMzRi0+lpognZdtcKYoiM+5Ix048CbSqxdz7IEyXVXqqxmr1Yg+UEcSWFZxFc6tBGP0GaChbuOpaayMYIsQUWdDy0f5Noe7MM59BpNmnHnJbNRY7GKqQ+s2KwZTAYmjpld0z62ERYm1fJS8V+B7J+CjD9TLk8kfEp4AfnMZitfSCqoLjRDuf+oX/0FyyfKeo5zIwJ6YQmySso1xE3HOVaGqxGf4NcLfwLGR4aJnohtOi2L9LgR3nE/8JEzcRKzwqFgpIVa2vKE3JQS9vA7QV4EoTGSGN9AB4X/TWIj6vNDw/VQHXM0TYxEcSgpt6fyHx9JxQxk8a3JTvncGnZWjLdk66dwgGRUMvzTgzPq62pSbqiTvCfBDiH8tcUtt3UonRO9gfXmKJXTTsIfyP229OaCMtsxoXXmNoM00j6kYcjZ7V6Jv+SVkBMScOVRGZyDHLMJ+WA98Phs5oX5aw5J0Ndn5/kuKAFaf4iuR55MafVHc7ytq7de/h6h/c0t1zYLGo2zoe8RLP7jpn61CPCZHSRdpAHmIe17dNA

`step.model_dump(exclude_none=True)`는 SDK 객체 하나를 평범한 Python dictionary로 바꿉니다. `exclude_none=True`는 값이 `None`인 불필요한 항목을 결과에서 제외한다는 뜻입니다.

```python
step                    # SDK가 만든 Step 객체
step.model_dump()       # Step 객체 → Python dictionary
```

### `store=True`로 대화 이어가기

수동 history 방식은 우리 코드가 전체 대화 기록을 보관하고 매번 다시 보냅니다. Interactions API에서 `store=True`를 사용하면 서버가 interaction을 저장하고, 다음 요청에서 이전 interaction의 ID만 전달해 대화를 이어갈 수 있습니다.

- `store=True`: 이번 요청과 응답을 서버에 저장합니다.
- `interaction.id`: 저장된 interaction의 고유 ID입니다.
- `previous_interaction_id`: 다음 요청이 어느 대화 다음에 이어지는지 지정합니다.

> 실제 서비스에서는 개인정보를 보내기 전에 데이터 저장 정책과 보관 기간을 확인해야 합니다.

In [8]:
# 첫 번째 요청을 서버에 저장합니다.
stored_turn1 = client.interactions.create(
    model=model,
    input="내 이름은 민수야. '확인'이라고만 답해.",
    store=True,
)
print("첫 번째 답변:", stored_turn1.output_text)
print("interaction ID:", stored_turn1.id)

# 전체 history 대신 이전 interaction의 ID를 전달합니다.
stored_turn2 = client.interactions.create(
    model=model,
    input="내 이름이 뭐야? 이름만 답해.",
    previous_interaction_id=stored_turn1.id,
    store=True,
)
print("두 번째 답변:", stored_turn2.output_text)

첫 번째 답변: 확인
interaction ID: v1_ChYzek9OYW9pVkZJaW4ycm9Qai1MNGVBEhYzek9OYW9pVkZJaW4ycm9Qai1MNGVB
두 번째 답변: 민수


### 이전 시점에서 새로운 대화로 분기하기

`previous_interaction_id`에는 가장 최근 ID만 넣어야 하는 것이 아닙니다. 이전 interaction의 ID를 다시 사용하면 그 시점까지의 맥락에서 새로운 대화를 시작할 수 있습니다.

아래 예제에서는 1번에서 이름을 알려주고, 2번에서 좋아하는 음식 정보를 추가합니다. 그런 다음 1번으로 돌아가 새 대화를 분기하면 2번에서 추가한 정보는 전달되지 않습니다. 기존의 2번 대화가 삭제되는 것은 아닙니다.

In [9]:
# 1. 이름 알려주기
branch_turn1 = client.interactions.create(
    model=model,
    input="내 이름은 민수야.",
    store=True,
)
print("1번:", branch_turn1.output_text)

# 2. 이름을 물어보고 새로운 정보 추가하기
branch_turn2 = client.interactions.create(
    model=model,
    input="내 이름이 뭐야? 그리고 내가 좋아하는 음식은 피자야. 두 내용을 확인해줘.",
    previous_interaction_id=branch_turn1.id,
    store=True,
)
print("2번:", branch_turn2.output_text)

# 3. 2번이 아닌 1번에서 분기해 좋아하는 음식 물어보기
branch_from_turn1 = client.interactions.create(
    model=model,
    input="내 이름과 내가 좋아하는 음식이 뭐야?",
    previous_interaction_id=branch_turn1.id,
    store=True,
)
print("1번에서 새로 분기:", branch_from_turn1.output_text)

1번: 안녕하세요, 민수 님! 만나서 반갑습니다. 

오늘 어떤 도움이 필요하신가요? 편하게 말씀해 주세요!
2번: 네, 확인해 드릴게요!

1. **이름:** **민수** 님
2. **좋아하는 음식:** **피자**

민수 님의 이름과 좋아하는 음식(피자)을 잘 기억해 둘게요! 피자 종류 중에서는 어떤 피자를 가장 좋아하시나요?
1번에서 새로 분기: 이름은 **민수** 님이에요! 

하지만 **좋아하는 음식**은 아직 말씀해 주지 않으셔서 제가 모르고 있어요. 민수 님이 어떤 음식을 가장 좋아하는지 알려주시면 기억해 둘게요! 

어떤 음식을 좋아하시나요?


### 대화 맥락의 핵심

- history 없이 물으면 이름 정보가 없으므로 `모름`이라고 답합니다.
- 이름이 들어 있는 history를 함께 보내면 `민수`라고 답합니다.
- 모델 자체가 개인을 영구적으로 기억하는 것은 아닙니다.
- 수동 history 방식에서는 앱이 `history` 전체를 매번 다시 보냅니다.
- `store=True` 방식에서는 서버가 대화를 저장하고 ID로 연결합니다.
- 이전 interaction의 ID를 다시 지정하면 그 시점에서 새로운 대화로 분기할 수 있습니다.
- 대화가 길어지면 토큰, 비용과 지연 시간이 늘 수 있습니다.
- 실제 서비스에서는 대화 기록의 길이, 개인정보와 서버 저장 정책을 관리해야 합니다.

## Generation config

프롬프트는 모델에게 원하는 답변 방식을 설명하고, generation_config는 답변을 생성하는 방법을 코드로 설정합니다.

- `max_output_tokens`: 생성할 수 있는 최대 token 수입니다. 정확한 글자 수가 아니며, 너무 작으면 답변이 중간에 끝날 수 있습니다.
- `thinking_level`: 모델이 답변 전에 사용하는 추론 수준입니다. `low`, `medium`, `high` 중에서 선택합니다.

> 같은 설정이어도 답변이 항상 완전히 같아지는 것은 아닙니다.

In [10]:
configured = client.interactions.create(
    model=model,
    input="LLM API를 한 문장으로 설명해줘.",
    generation_config={
        "max_output_tokens": 30,
    },
    store=False,
)
print(configured.output_text)

## 실패도 정상적인 흐름이다

API 호출 실패를 모두 같은 오류로 처리하면 원인과 대응 방법을 알 수 없습니다. Google GenAI SDK는 400번대 응답을 `ClientError`, 500번대 응답을 `ServerError`로 구분합니다.

| 오류 | 예 | 기본 대응 |
|---|---|---|
| `ClientError` | 잘못된 API 키, 모델 이름, 요청 형식 | 요청이나 설정을 수정해야 합니다. |
| `ClientError` 429 | 사용량 제한 | 사용량과 제한을 확인하고 잠시 후 다시 요청합니다. |
| `ServerError` | Gemini 서버의 일시적인 500번대 오류 | 잠시 후 다시 요청합니다. |

In [11]:
def ask_gemini(prompt: str) -> str | None:
    try:
        result = client.interactions.create(
            model=model, input=prompt, store=False,
        )
        return result.output_text
    except errors.ClientError as error:
        if error.code in [401, 403]:
            print("인증 실패: API 키와 권한을 확인하세요.")
        elif error.code == 404:
            print("요청한 모델을 찾을 수 없습니다.")
        elif error.code == 429:
            print("사용량 제한에 도달했습니다. 잠시 후 다시 요청하세요.")
        else:
            print(f"요청 오류({error.code}): {error.message}")
        return None
    except errors.ServerError as error:
        print(f"서버 오류({error.code}): 잠시 후 다시 요청하세요.")
        return None

answer = ask_gemini("Python 함수를 30자 이내의 한 문장으로 설명해줘.")
if answer:
    print(answer)

파이썬 함수는 재사용 가능한 코드 블록입니다.


## 종합 실습: 대화형 여행 플래너 만들기

앞에서 배운 기능을 조합해 대화형 여행 플래너를 만들어 보세요.

여행지를 입력받아 첫 여행 일정을 만든 뒤, 사용자가 원하는 만큼 추가 요청이나 수정 의견을 반영해 보세요. 앞에서 나눈 내용을 기억한 상태로 답해야 하며, 사용자가 `종료`를 입력하면 여행 계획을 마칩니다.

In [12]:
# 1. 여행지 입력받기
destination = input("여행지를 입력하세요")

branch_turn1 = client.interactions.create(
    model=model,
    input=f"나는 {destination} 여행을 계획할 거야. 일정을 추천해줘.",
    store=True,
)
print("추천:", branch_turn1.output_text)

추천: 도쿄 여행은 동선이 매우 중요합니다. 가장 대중적이고 알찬 **3박 4일 일정**을 추천해 드릴게요! 

처음 도쿄를 가시거나, 핵심 명소를 모두 둘러보고 싶은 분들에게 최적화된 코스입니다.

---

### 🗓️ [3박 4일] 도쿄 핵심 정복 추천 일정

#### **1일 차: 도쿄의 전통과 현대 (동부 도쿄)**
*입국 후 숙소 체크인 후 시작*
*   **오후: 아사쿠사 (센소지)**
    *   도쿄에서 가장 오래된 절과 전통 거리인 '나카미세도리'에서 길거리 음식(인형焼き, 멜론빵 등) 맛보기.
*   **늦은 오후: 도쿄 스카이트리**
    *   아사쿠사에서 도보/쇼핑몰 이동 가능. 일본에서 가장 높은 타워에서 도쿄 전경 감상 및 대형 쇼핑몰(소라마치) 구경.
*   **저녁: 우에노 또는 아키하바라**
    *   **우에노:** '아메요코 시장'에서 시끌벅적한 이자카야 감성 즐기기.
    *   **아키하바라:** 애니메이션, 피규어, 전자제품에 관심이 있다면 필수 방문.

#### **2일 차: 트렌드, 쇼핑, 그리고 야경 (서부 도쿄)**
*   **오전: 메이지 신궁 & 하라주쿠**
    *   도심 속 거대한 숲인 '메이지 신궁' 산책 후, 개성 넘치는 '다케시타 도리' 거리 구경.
*   **오후: 오모테산도 & 캣스트리트**
    *   '도쿄의 샹젤리제'라 불리는 명품 거리와 예쁜 카페, 스트릿 브랜드(베이프, 슈프림 등) 쇼핑.
*   **저녁: 시부야**
    *   유명한 **'시부야 스크램블 교차로'** 건너보기.
    *   **'시부야 스카이'** 전망대에서 도쿄 최고의 일몰과 야경 감상 (*사전 예약 필수!*).
    *   시부야 근처에서 미소라멘이나 스시로 저녁 식사.

#### **3일 차: 미식과 감성, 그리고 도쿄 타워 (중부 도쿄)**
*   **오전: 츠키지 장외시장**
    *   신선한 해산물 덮밥(카이센동), 계란말이, 호르몬동(곱창덮밥) 등 아침 겸 점심 식사.
*   **오

In [14]:
previous_interaction_id = None

while True :
    user_input = input("대화를 입력하세요")

    if user_input.lower() in ["종료", "exit"] :
        break
    options = {
        "model" : model,
        "input" : user_input,
        "store" : True
    }

    if previous_interaction_id :
        options["previous_interaction_id"] = previous_interaction_id
    interaction = client.interactions.create(**options)

    print("AI:", interaction.output_text)

    # 다음 대화가 현재 대화를 이어가도록 ID 갱신
    previous_interaction_id = interaction.id

AI: 전통과 현대가 매력적으로 어우러진 도쿄 필수 여행지 4곳을 추천합니다.

1. **아사쿠사 (센소지)**: 도쿄에서 가장 오래된 사찰로, 일본 특유의 고풍스러운 분위기를 느낄 수 있습니다. 입구인 '카미나리몬'과 상점가 '나카미세 도리'에서 맛있는 주전부리를 즐겨보세요.
2. **시부야 (시부야 스카이)**: 초고층 전망대에서 도쿄 도심과 유명한 스크램블 교차로를 한눈에 담을 수 있습니다. 특히 일몰과 야경이 환상적입니다. (예약 필수)
3. **신주쿠 교엔**: 도심 한가운데 위치한 넓고 아름다운 정원입니다. 화려한 도시 여행 중 잠시 쉬어가며 여유롭게 산책하기 좋습니다.
4. **롯폰기 힐즈**: 도쿄의 상징인 '도쿄 타워'를 가장 예쁘게 담을 수 있는 조망 명소입니다. 미술관과 쇼핑몰이 모여 있어 문화생활도 함께 즐길 수 있습니다.

취향에 맞춰 알찬 도쿄 여행을 즐겨보세요!
AI: 추천해 드린 4곳은 도쿄의 동쪽에서 서쪽, 그리고 중앙으로 이어지는 위치에 있어 **하루(1일) 만에 동선을 최적화하여 효율적으로 둘러볼 수 있습니다.** 

지하철 이동 시간을 최소화한 **'알찬 하루 코스'**를 안내해 드립니다.

---

### 🗺️ 도쿄 핵심 1일 추천 경로

**① [오전 09:00] 아사쿠사 (센소지)**
*   **일정:** 사람이 붐비기 전 고즈넉한 전통 사찰 둘러보기 & '나카미세 도리'에서 실크 푸딩, 당고 등 주전부리 맛보기.
*   *이동:* 아사쿠사역 ➔ 신주쿠교엔마에역 (지하철 약 30분)

**② [점심 12:00] 신주쿠 교엔**
*   **일정:** 신주쿠 근처에서 점심 식사 후, 넓은 정원을 산책하며 휴식과 사진 촬영.
*   *이동:* 신주쿠산초메역 ➔ 시부야역 (지하철 약 10분)

**③ [오후 15:30] 시부야 (시부야 스카이)**
*   **일정:** 시부야 거리 쇼핑 & 스크램블 교차로 체험 ➔ **시부야 스카이 전망대**에서 노을 감상 (일몰 시간 전 입장을 추천!).
*   *이동:* 시부야역 ➔ 롯폰기